# Data Exploration: MNIST PU Dataset

This notebook explores the MNIST Positive-Unlabeled (PU) dataset used in the nnPU paper reproduction.

**Dataset Configuration:**
- Positive class: Even digits (0, 2, 4, 6, 8)
- Negative class: Odd digits (1, 3, 5, 7, 9)
- Labeled positive samples: 100
- Unlabeled samples: ~59,900 (contains both positive and negative)
- Class prior π: ~0.49

In [ ]:
import sys
from pathlib import Path

# Add src to path
sys.path.insert(0, str(Path.cwd().parent / "src"))

import matplotlib.pyplot as plt
from pu_learning.data.mnist_pu import MNISTPUDataset
from torch.utils.data import DataLoader

from pu_learning.utils.reproducibility import set_seed

# Set reproducible seed
set_seed(42)

## 1. Load Dataset

In [ ]:
# Create dataset with default configuration
train_dataset = MNISTPUDataset(
    root="../data",
    train=True,
    positive_class="even",
    n_labeled=100,
    download=True,
)

test_dataset = MNISTPUDataset(
    root="../data",
    train=False,
    positive_class="even",
    download=True,
)

print(f"Training set size: {len(train_dataset):,}")
print(f"Test set size: {len(test_dataset):,}")
print(f"Class prior π: {train_dataset.class_prior:.4f}")

## 2. Dataset Statistics

In [ ]:
# Count labeled positive, unlabeled positive, unlabeled negative
n_labeled_pos = sum(1 for i in range(len(train_dataset)) if train_dataset.pu_labels[i] == 1)
n_unlabeled = sum(1 for i in range(len(train_dataset)) if train_dataset.pu_labels[i] == 0)

# Get true labels for unlabeled set
unlabeled_indices = [i for i in range(len(train_dataset)) if train_dataset.pu_labels[i] == 0]
n_unlabeled_pos = sum(1 for i in unlabeled_indices if train_dataset.true_labels[i] == 1)
n_unlabeled_neg = n_unlabeled - n_unlabeled_pos

print("Training Set Breakdown:")
print(f"  Labeled Positive (P): {n_labeled_pos:,}")
print(f"  Unlabeled (U): {n_unlabeled:,}")
print(f"    - True Positive: {n_unlabeled_pos:,}")
print(f"    - True Negative: {n_unlabeled_neg:,}")
print(f"\nTotal True Positive: {n_labeled_pos + n_unlabeled_pos:,}")
print(f"Total True Negative: {n_unlabeled_neg:,}")
print(f"Estimated class prior: {(n_labeled_pos + n_unlabeled_pos) / len(train_dataset):.4f}")

## 3. Visualize Sample Images

In [ ]:
# Get some labeled positive samples
labeled_pos_indices = [i for i in range(len(train_dataset)) if train_dataset.pu_labels[i] == 1]

# Get some unlabeled samples (with true labels for visualization)
unlabeled_pos_indices = [i for i in unlabeled_indices if train_dataset.true_labels[i] == 1][:5]
unlabeled_neg_indices = [i for i in unlabeled_indices if train_dataset.true_labels[i] == 0][:5]

fig, axes = plt.subplots(3, 5, figsize=(12, 8))
fig.suptitle("Sample Images from MNIST PU Dataset", fontsize=16)

# Row 1: Labeled Positive
for idx, ax in enumerate(axes[0]):
    img, pu_label, true_label = train_dataset[labeled_pos_indices[idx]]
    ax.imshow(img.squeeze(), cmap="gray")
    ax.set_title(f"Labeled P\nDigit: {train_dataset.mnist_dataset.targets[labeled_pos_indices[idx]]}")
    ax.axis("off")

# Row 2: Unlabeled (True Positive)
for idx, ax in enumerate(axes[1]):
    img, pu_label, true_label = train_dataset[unlabeled_pos_indices[idx]]
    ax.imshow(img.squeeze(), cmap="gray")
    ax.set_title(f"Unlabeled\nDigit: {train_dataset.mnist_dataset.targets[unlabeled_pos_indices[idx]]} (Even)")
    ax.axis("off")

# Row 3: Unlabeled (True Negative)
for idx, ax in enumerate(axes[2]):
    img, pu_label, true_label = train_dataset[unlabeled_neg_indices[idx]]
    ax.imshow(img.squeeze(), cmap="gray")
    ax.set_title(f"Unlabeled\nDigit: {train_dataset.mnist_dataset.targets[unlabeled_neg_indices[idx]]} (Odd)")
    ax.axis("off")

plt.tight_layout()
plt.show()

## 4. Distribution of Digits

In [ ]:
# Count digits in each category
from collections import defaultdict

digit_counts = {
    "labeled_pos": defaultdict(int),
    "unlabeled_pos": defaultdict(int),
    "unlabeled_neg": defaultdict(int),
}

for idx in labeled_pos_indices:
    digit = train_dataset.mnist_dataset.targets[idx].item()
    digit_counts["labeled_pos"][digit] += 1

for idx in unlabeled_indices:
    digit = train_dataset.mnist_dataset.targets[idx].item()
    if train_dataset.true_labels[idx] == 1:
        digit_counts["unlabeled_pos"][digit] += 1
    else:
        digit_counts["unlabeled_neg"][digit] += 1

# Plot distribution
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle("Digit Distribution in Training Set", fontsize=16)

categories = ["labeled_pos", "unlabeled_pos", "unlabeled_neg"]
titles = ["Labeled Positive (100)", "Unlabeled (True Positive)", "Unlabeled (True Negative)"]

for ax, category, title in zip(axes, categories, titles):
    digits = sorted(digit_counts[category].keys())
    counts = [digit_counts[category][d] for d in digits]

    ax.bar(digits, counts)
    ax.set_xlabel("Digit")
    ax.set_ylabel("Count")
    ax.set_title(title)
    ax.set_xticks(range(10))
    ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Test Set Distribution

In [ ]:
# Count even vs odd in test set
n_test_pos = sum(1 for i in range(len(test_dataset)) if test_dataset.true_labels[i] == 1)
n_test_neg = len(test_dataset) - n_test_pos

print("Test Set Breakdown:")
print(f"  Even digits (Positive): {n_test_pos:,}")
print(f"  Odd digits (Negative): {n_test_neg:,}")
print(f"  Class balance: {n_test_pos / len(test_dataset):.4f}")

# Plot test set distribution by digit
test_digit_counts = defaultdict(int)
for idx in range(len(test_dataset)):
    digit = test_dataset.mnist_dataset.targets[idx].item()
    test_digit_counts[digit] += 1

fig, ax = plt.subplots(figsize=(10, 5))
digits = sorted(test_digit_counts.keys())
counts = [test_digit_counts[d] for d in digits]
colors = ["blue" if d % 2 == 0 else "orange" for d in digits]

ax.bar(digits, counts, color=colors, alpha=0.7)
ax.set_xlabel("Digit")
ax.set_ylabel("Count")
ax.set_title("Test Set Digit Distribution (Blue=Even/Positive, Orange=Odd/Negative)")
ax.set_xticks(range(10))
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Data Loader Example

In [ ]:
# Create data loaders
train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False)

print(f"Number of training batches: {len(train_loader)}")
print(f"Number of test batches: {len(test_loader)}")

# Get a batch
images, pu_labels, true_labels = next(iter(train_loader))

print("\nBatch shapes:")
print(f"  Images: {images.shape}")
print(f"  PU labels: {pu_labels.shape}")
print(f"  True labels: {true_labels.shape}")
print("\nBatch statistics:")
print(f"  Labeled positive in batch: {pu_labels.sum().item()}")
print(f"  Unlabeled in batch: {(pu_labels == 0).sum().item()}")
print(f"  True positive in batch: {true_labels.sum().item()}")

## Summary

This notebook explored the MNIST PU dataset configuration used for reproducing the nnPU paper:

1. **Training set**: 100 labeled positive + ~59,900 unlabeled samples
2. **Test set**: ~10,000 fully labeled samples
3. **Positive class**: Even digits (0, 2, 4, 6, 8)
4. **Negative class**: Odd digits (1, 3, 5, 7, 9)
5. **Class prior π**: ~0.49 (approximately balanced)

The dataset is ready for training PN, uPU, and nnPU models!